# Diffusion Models — DDPM from Scratch Notebook

> Hands-on Build It and Exercises.

## Build It

`code/main.py` implements a 1-D DDPM. Data is a two-mode mixture. The "net" is a tiny MLP that takes `(x_t, t)` and outputs predicted noise. Training is the one-line loss. Sampling iterates the reverse chain.

### Step 1: the forward schedule (closed form)

In [ ]:
```python

betas = [1e-4 + (0.02 - 1e-4) * t / (T - 1) for t in range(T)]

alphas = [1 - b for b in betas]

alpha_bars = []

cum = 1.0

for a in alphas:

    cum *= a

    alpha_bars.append(cum)

In [ ]:
```

### Step 2: sample `x_t` in one shot

In [ ]:
```python

def forward_sample(x0, t, alpha_bars, rng):

    a_bar = alpha_bars[t]

    eps = rng.gauss(0, 1)

    x_t = math.sqrt(a_bar) * x0 + math.sqrt(1 - a_bar) * eps

    return x_t, eps

In [ ]:
```

### Step 3: one training step

In [ ]:
```python

def train_step(x0, model, alpha_bars, rng):

    t = rng.randrange(T)

    x_t, eps = forward_sample(x0, t, alpha_bars, rng)

    eps_hat = model_forward(model, x_t, t)

    loss = (eps - eps_hat) ** 2

    return loss, gradient_step(model, ...)

In [ ]:
```

### Step 4: reverse sampling

In [ ]:
```python

def sample(model, alpha_bars, T, rng):

    x = rng.gauss(0, 1)

    for t in range(T - 1, -1, -1):

        eps_hat = model_forward(model, x, t)

        beta_t = 1 - alphas[t]

        x = (x - beta_t / math.sqrt(1 - alpha_bars[t]) * eps_hat) / math.sqrt(alphas[t])

        if t > 0:

            x += math.sqrt(beta_t) * rng.gauss(0, 1)

    return x

In [ ]:
```

For a 1-D problem with 40 timesteps and a 24-unit MLP, this learns the two-mode mixture in ~200 epochs.

## Exercises

In [ ]:
1. **Easy.** Change T from 40 to 10 in `code/main.py`. How does sample quality (visual histogram of outputs) degrade? At what T does the two-mode structure collapse?
2. **Medium.** Switch from ε-prediction to v-prediction. Re-derive the reverse step. Compare final sample quality.
3. **Hard.** Add classifier-free guidance. Condition on a class label `c ∈ {0, 1}`, drop it 10% of the time during training, and at sampling time use `ε = (1+w)·ε_cond - w·ε_uncond`. Measure the conditional-mode-hit rate at `w = 0, 1, 3, 7`.